# FireWatch: train and evaluate the YOLO detector on a Colab GPU

Runs the DVC stages `ingest → validate → prepare → prepare_simuletic → train_detector → evaluate_detector` and `compare`, plus a second detector configuration, logged to MLflow on DagsHub. It then registers the best detector in **Staging**.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Add these Colab secrets (key icon on the left), with notebook access switched on:

| Secret | Value |
|---|---|
| `KAGGLE_USERNAME`, `KAGGLE_KEY` | from your `kaggle.json` |
| `DAGSHUB_USER`, `DAGSHUB_TOKEN` | DagsHub username and token (DagsHub → Settings → Tokens) |

At the end, download `detector_results.zip` (last cell) and give it back to commit `dvc.lock` and the reports from your machine. Model weights go to the DVC remote from here.

In [ ]:
!nvidia-smi -L
BRANCH = "feat/detector"
!git clone -q -b $BRANCH https://github.com/ushnahkhan/firewatch.git /content/firewatch
%cd /content/firewatch
!pip install -q -r requirements.txt

In [ ]:
import os, json, pathlib
from google.colab import userdata
for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "DAGSHUB_USER", "DAGSHUB_TOKEN"):
    os.environ[k] = userdata.get(k)
kdir = pathlib.Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
(kdir / "kaggle.json").write_text(json.dumps({"username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}))
os.chmod(kdir / "kaggle.json", 0o600)
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/ushnahkhan/firewatch.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ["DAGSHUB_USER"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["DAGSHUB_TOKEN"]
# DVC credentials go to .dvc/config.local (never committed)
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user "$DAGSHUB_USER"
!dvc remote modify origin --local password "$DAGSHUB_TOKEN"
!python -m src.tracking.mlflow_utils   # smoke test: one tiny run in firewatch-detector

## 1. Data stages and config A (`detector-v1`) through DVC
`dvc repro evaluate_detector` runs only that stage and its upstream stages. The baseline stages are not touched: their committed metrics are used as they are.

In [ ]:
!cp dvc.lock /content/dvc.lock.orig
!dvc repro evaluate_detector

In [ ]:
# The data stages must give the same hashes as the committed dvc.lock (same data as on the DVC remote)
import yaml
old = yaml.safe_load(open("/content/dvc.lock.orig"))["stages"]
new = yaml.safe_load(open("dvc.lock"))["stages"]
for st in ("ingest", "validate", "validate_simuletic", "prepare", "prepare_simuletic"):
    a = {o["path"]: o.get("md5") for o in old[st].get("outs", [])}
    b = {o["path"]: o.get("md5") for o in new[st].get("outs", [])}
    print(st, "same" if a == b else f"CHANGED {a} -> {b}")

## 2. Config B (`detector-lr2e-3`), run directly with a lower learning rate

In [ ]:
!python -m src.detector.train --set detector.lr0=0.002 --set detector.run_name=detector-lr2e-3 --set detector.out_dir=models/detector_lr2e-3
!python -m src.detector.evaluate --set detector.out_dir=models/detector_lr2e-3

## 3. Compare with the baseline, register the best detector (Staging), push weights

In [ ]:
!dvc repro -s compare
!python -m src.tracking.register
!dvc push train_detector evaluate_detector
!dvc status train_detector evaluate_detector compare

In [ ]:
!zip -q detector_results.zip dvc.lock params.yaml reports/detector_metrics.json reports/detector_metrics_detector_lr2e-3.json      reports/detector_train_detector.json reports/detector_train_detector_lr2e-3.json reports/comparison.md reports/comparison.png      -r reports/detector_eval_detector reports/detector_eval_detector_lr2e-3 models/detector/meta.json models/detector/results.png      models/detector_lr2e-3/meta.json models/detector_lr2e-3/results.png
from google.colab import files
files.download("detector_results.zip")